# Build an Email Agent with AgentMail

This notebook shows how to build an AI agent that can send, receive, and reply to emails using [AgentMail](https://agentmail.to) — the email API built for AI agents.

Unlike traditional email APIs (SendGrid, SES) which focus on one-way sending, AgentMail gives agents their own email inboxes with full two-way communication: send, receive, reply, forward, manage threads, and process attachments.

## What you'll build

An AI customer support agent that:
1. Gets its own email inbox
2. Receives customer emails
3. Uses GPT-4 to understand the request and generate a response
4. Replies to the customer via email

## Prerequisites

- OpenAI API key
- AgentMail API key (free at [console.agentmail.to](https://console.agentmail.to))

In [ ]:
!pip install openai agentmail

In [ ]:
import os
from openai import OpenAI
from agentmail import AgentMail

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
agentmail_client = AgentMail(api_key=os.environ["AGENTMAIL_API_KEY"])

## Step 1: Create an inbox for your agent

Each agent gets its own unique email address. You can create as many inboxes as you need — one per agent, per customer, or per conversation.

In [ ]:
inbox = agentmail_client.inboxes.create(
    display_name="Support Agent"
)
print(f"Agent email: {inbox.inbox_id}")

## Step 2: Send an email from your agent

Your agent can send emails to anyone. The email comes from the agent's own inbox address.

In [ ]:
message = agentmail_client.inboxes.messages.send(
    inbox_id=inbox.inbox_id,
    to="customer@example.com",
    subject="Welcome! How can I help?",
    text="Hi there! I'm your AI support agent. Reply to this email with any questions and I'll help you out."
)
print(f"Sent message: {message.message_id}")
print(f"Thread: {message.thread_id}")

## Step 3: Check for incoming emails

List messages in the inbox to see replies. In production, you'd use webhooks or WebSockets for real-time processing.

In [ ]:
messages = agentmail_client.inboxes.messages.list(inbox_id=inbox.inbox_id)

for msg in messages.messages:
    print(f"From: {msg.from_}")
    print(f"Subject: {msg.subject}")
    # Use extracted_text to get just the new content (strips quoted replies)
    content = msg.extracted_text or msg.text
    print(f"Content: {content}")
    print("---")

## Step 4: Build the AI email agent

Now let's combine OpenAI and AgentMail into a working support agent. This function:
1. Reads new incoming emails
2. Uses GPT-4 to generate a contextual reply
3. Sends the reply via AgentMail
4. Uses `extracted_text` to handle email chains cleanly

In [ ]:
SYSTEM_PROMPT = """You are a helpful customer support agent for a SaaS product.
Reply to customer emails concisely and helpfully.
If you don't know the answer, say so and offer to escalate.
Keep replies under 3 paragraphs."""

def process_email(inbox_id: str, message_id: str):
    """Process an incoming email and send an AI-generated reply."""
    
    # Get the message
    msg = agentmail_client.inboxes.messages.get(
        inbox_id=inbox_id,
        message_id=message_id
    )
    
    # Use extracted_text to get just the new content from the reply
    customer_message = msg.extracted_text or msg.text
    
    # Get thread history for context
    thread = agentmail_client.inboxes.threads.get(
        inbox_id=inbox_id,
        thread_id=msg.thread_id
    )
    
    # Build conversation history for GPT
    conversation = []
    for thread_msg in thread.messages:
        role = "assistant" if thread_msg.from_ == inbox_id else "user"
        content = thread_msg.extracted_text or thread_msg.text
        conversation.append({"role": role, "content": content})
    
    # Generate reply with GPT-4
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            *conversation
        ]
    )
    
    reply_text = response.choices[0].message.content
    
    # Send the reply via AgentMail
    reply = agentmail_client.inboxes.messages.reply(
        inbox_id=inbox_id,
        message_id=message_id,
        text=reply_text
    )
    
    print(f"Replied to {msg.from_}: {reply_text[:100]}...")
    return reply

## Step 5: Process all unread emails

In production, you'd trigger this via webhooks. Here we'll poll for new messages.

In [ ]:
messages = agentmail_client.inboxes.messages.list(inbox_id=inbox.inbox_id)

for msg in messages.messages:
    # Skip messages sent by our agent
    if msg.from_ == inbox.inbox_id:
        continue
    
    print(f"Processing email from {msg.from_}: {msg.subject}")
    process_email(inbox.inbox_id, msg.message_id)

## Using OpenAI Function Calling with AgentMail

You can also use AgentMail as a tool with OpenAI's function calling. The `agentmail-toolkit` package provides ready-made tool definitions.

In [ ]:
!pip install agentmail-toolkit

In [ ]:
from agentmail_toolkit import AgentMailToolkit

toolkit = AgentMailToolkit(api_key=os.environ["AGENTMAIL_API_KEY"])

# Get OpenAI-compatible tool definitions
tools = toolkit.get_openai_tools()

print(f"Available tools: {[t['function']['name'] for t in tools]}")

In [ ]:
# Let GPT-4 use AgentMail tools autonomously
response = openai_client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are an AI assistant with email capabilities. Use the email tools to help the user."},
        {"role": "user", "content": "Create a new email inbox and send a welcome email to test@example.com"}
    ],
    tools=tools
)

# Process tool calls
for tool_call in response.choices[0].message.tool_calls or []:
    print(f"Tool: {tool_call.function.name}")
    print(f"Args: {tool_call.function.arguments}")
    
    # Execute the tool
    result = toolkit.execute_tool(tool_call.function.name, tool_call.function.arguments)
    print(f"Result: {result}")

## Production Setup: Webhooks

For production, use webhooks instead of polling. AgentMail sends events to your endpoint in real-time.

```python
# Create a webhook
webhook = agentmail_client.webhooks.create(
    url="https://your-server.com/webhook",
    events=["message.received"]
)

# Your webhook handler (Flask example)
from flask import Flask, request

app = Flask(__name__)

@app.route('/webhook', methods=['POST'])
def handle_webhook():
    event = request.json
    if event['type'] == 'message.received':
        process_email(
            inbox_id=event['data']['inbox_id'],
            message_id=event['data']['message_id']
        )
    return '', 200
```

## Next Steps

- **Custom domains:** Use `agentmail_client.domains.create()` to send from your own domain
- **Attachments:** Send and receive files with messages
- **Semantic search:** Search across inboxes using natural language
- **WebSockets:** Real-time events without a public endpoint
- **Pods:** Isolate email environments per customer

Learn more at [docs.agentmail.to](https://docs.agentmail.to)